In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import iqr

#Data Load and Cleaning

In [ ]:
df = pd.read_csv('../data/operations.csv')
df = df.convert_dtypes()
# print("-------------\n Unique Values \n------------")
# print(df.nunique().sort_values(ascending=False))
df = df.drop(columns=['race']) #since it has only one unique value
# print(df.shape)
# print("-------------\n Null values \n------------")
# print((df.isnull().mean().mul(100).round(2)).sort_values(ascending=False))
df = df.drop(columns=['cpbon_time','cpboff_time','icuin_time','icuout_time','inhosp_death_time','allcause_death_time','case_id']) #Missig values in these columns are more than 80%
# print(df1.isnull().mean().mul(100).round(2).sort_values(ascending=False))
df = df.dropna(how='any', subset=['opend_time','opstart_time','anstart_time','anend_time']) # Data missing in these columns are less than 1 percent

#Data Manipulation

In [ ]:
# df[['height','weight','asa']].describe()
df['asa']=df['asa'].fillna(df['asa'].mode()[0]) #imputing mode as it is a ordinal column
df['height'] = df['height'].fillna(df['height'].median()) #imputing with median since the data is right skewed
df['weight'] = df['weight'].fillna(df['weight'].median()) #imputing with median since the data is right skedwe

In [ ]:
# df['height'].quantile([0.25,0.50,0.75])
ht_iqr = df['height'].quantile(0.75) - df['height'].quantile(0.25)
uplmt = df['height'].quantile(0.75) + 1.5 * ht_iqr
lolmt = df['height'].quantile(0.25) - 1.5 * ht_iqr
df['height'] = df['height'].astype('float64').round(2)
# Winsorization (best for clinical data)
df['height'] = df['height'].clip(lower=lolmt, upper=uplmt)

In [ ]:
wt_iqr = iqr(df['weight'])
wtup = df['weight'].quantile(0.75) + 1.5 * wt_iqr
wtlw = df['weight'].quantile(0.25) - 1.5 * wt_iqr
df['weight'] = df['weight'].astype('float64').round(2)
df['weight'] = df['weight'].clip(upper=wtup, lower=wtlw)

In [ ]:
# df.groupby('sex')['op_id'].count()
# df[['age','height','weight']].describe()

In [ ]:
cols = ['admission_time','orin_time','anstart_time','opstart_time','anend_time','opend_time','orout_time','discharge_time']
for col in cols:
  df[col] = pd.to_timedelta(df[col], unit='m')

In [ ]:
# df[['opdate','admission_time','orin_time','anstart_time','opstart_time','anend_time','opend_time','orout_time','discharge_time']].describe()

In [ ]:
df = df[df['opdate'] == 0] #keeping only the first admission or surgeries
outlier_indices = df[df['opstart_time'] >= pd.Timedelta(days=9)].index #removing outlier of 1 patient having 9 days before operation hospital stay
df = df.drop(outlier_indices)

In [ ]:
mask = df.groupby('subject_id')['hadm_id'].transform('count') == 1 # keeping only one encounter of the subject_id
df = df[mask]

In [ ]:
# df[['age','height','weight']].describe()
# df.groupby('sex')['op_id'].count()
# df.shape

Demographics are Stable:
Age: The mean only shifted from 55.7 to 57.5. This is a negligible change (about 3%), meaning you aren't accidentally excluding younger or older patients.
Height/Weight: The mean and standard deviation are almost identical. Your "average patient" in the 11k subset is physically the same as the "average patient" in the 130k set.
Gender Balance:
Before: ~56% Female / 44% Male.
After: ~55% Female / 45% Male.
The ratio remains consistent, which is crucial for medical research to avoid gender bias.
Range Preservation:
The Min and Max values for age (20 to 90), height, and weight haven't changed. This means kept the diversity of your population; you didn't accidentally chop off the "extremes."
